In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request

# Select the Ollama tag to use. Kaggle Internet must be enabled to download it.
OLLAMA_MODEL = 'llama3.1:8b'

# Attach a Kaggle Dataset with your checkpoint and fill in these two values.
COMFY_CHECKPOINT_DIR = Path('/kaggle/input/REPLACE_WITH_DATASET/checkpoints')
COMFY_CHECKPOINT_FILENAME = ''  # Example: 'my-checkpoint.safetensors'

WORK_ROOT = Path('/kaggle/working/rpgbot')
COMFYUI_DIR = WORK_ROOT / 'ComfyUI'
BRIDGE_ROOT = Path('/kaggle/working/bridge')
BRIDGE_INPUT = BRIDGE_ROOT / 'input'
BRIDGE_OUTPUT = BRIDGE_ROOT / 'output'
BRIDGE_ERRORS = BRIDGE_ROOT / 'errors'
LOG_DIR = WORK_ROOT / 'logs'
OLLAMA_HOST, OLLAMA_PORT = '127.0.0.1', 11434
COMFY_HOST, COMFY_PORT = '127.0.0.1', 8188

for directory in (WORK_ROOT, BRIDGE_INPUT, BRIDGE_OUTPUT, BRIDGE_ERRORS, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def run_command(command, *, cwd=None, env=None, input_text=None):
    result = subprocess.run(command, cwd=str(cwd) if cwd else None, env=env,
                            input=input_text, text=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT)
    if result.stdout:
        print(result.stdout[-12000:])
    if result.returncode:
        raise RuntimeError(f'Command failed ({result.returncode}): {command[0]}')
    return result

def http_json(url, *, method='GET', payload=None, timeout=30):
    data = None if payload is None else json.dumps(payload, ensure_ascii=False).encode('utf-8')
    request = urllib.request.Request(url, data=data,
        headers={'Content-Type': 'application/json'} if data is not None else {}, method=method)
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode('utf-8'))

if not OLLAMA_MODEL.strip():
    raise ValueError('Set OLLAMA_MODEL before starting.')
if not COMFY_CHECKPOINT_FILENAME.strip():
    raise ValueError('Set COMFY_CHECKPOINT_FILENAME before starting.')
if not COMFY_CHECKPOINT_DIR.is_dir():
    raise FileNotFoundError(f'Checkpoint directory does not exist: {COMFY_CHECKPOINT_DIR}')
checkpoint_path = COMFY_CHECKPOINT_DIR / COMFY_CHECKPOINT_FILENAME
if not checkpoint_path.is_file():
    raise FileNotFoundError(f'Checkpoint file does not exist: {checkpoint_path}')

print(f'Upload Ollama requests to: {BRIDGE_INPUT}')
print(f'Retrieve same-name responses from: {BRIDGE_OUTPUT}')
print(f'ComfyUI checkpoint: {checkpoint_path}')

In [ ]:
# Install Ollama using the official Linux installer if needed.
if shutil.which('ollama') is None:
    with urllib.request.urlopen('https://ollama.com/install.sh', timeout=60) as response:
        installer = response.read().decode('utf-8')
    run_command(['bash'], input_text=installer)

if shutil.which('git') is None:
    raise RuntimeError('git is required to download ComfyUI.')
if not COMFYUI_DIR.exists():
    COMFYUI_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_command(['git', 'clone', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFYUI_DIR)])

# Link the Kaggle Dataset file; do not duplicate a potentially large checkpoint.
checkpoint_link = COMFYUI_DIR / 'models' / 'checkpoints' / COMFY_CHECKPOINT_FILENAME
checkpoint_link.parent.mkdir(parents=True, exist_ok=True)
if checkpoint_link.exists() or checkpoint_link.is_symlink():
    if checkpoint_link.resolve() != checkpoint_path.resolve():
        raise FileExistsError(f'Checkpoint link already points elsewhere: {checkpoint_link}')
else:
    checkpoint_link.symlink_to(checkpoint_path.resolve())

run_command([sys.executable, '-m', 'pip', 'install', '-r', str(COMFYUI_DIR / 'requirements.txt')])

# Keep both APIs running. Ollama releases its model after each inference to
# reduce VRAM contention with ComfyUI; ComfyUI runs with low-VRAM options.
ollama_env = os.environ.copy()
ollama_env.update({'OLLAMA_HOST': f'{OLLAMA_HOST}:{OLLAMA_PORT}',
                   'OLLAMA_MODELS': str(WORK_ROOT / 'ollama-models'),
                   'OLLAMA_KEEP_ALIVE': '0', 'OLLAMA_NUM_PARALLEL': '1',
                   'OLLAMA_MAX_LOADED_MODELS': '1'})
(WORK_ROOT / 'ollama-models').mkdir(parents=True, exist_ok=True)
ollama_log = open(LOG_DIR / 'ollama.log', 'a', encoding='utf-8')
ollama_process = subprocess.Popen(['ollama', 'serve'], env=ollama_env,
    stdout=ollama_log, stderr=subprocess.STDOUT, start_new_session=True)
run_command(['ollama', 'pull', OLLAMA_MODEL], env=ollama_env)

comfy_log = open(LOG_DIR / 'comfyui.log', 'a', encoding='utf-8')
comfy_process = subprocess.Popen([sys.executable, 'main.py', '--listen', COMFY_HOST,
    '--port', str(COMFY_PORT), '--lowvram', '--cpu-vae'], cwd=str(COMFYUI_DIR),
    stdout=comfy_log, stderr=subprocess.STDOUT, start_new_session=True)

def wait_for_service(name, url, process, log_name, timeout_seconds=600):
    deadline = time.monotonic() + timeout_seconds
    last_error = None
    while time.monotonic() < deadline:
        if process.poll() is not None:
            tail = (LOG_DIR / log_name).read_text(encoding='utf-8', errors='replace')[-6000:]
            raise RuntimeError(f'{name} exited ({process.returncode}). Recent log:\n{tail}')
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                if response.status == 200:
                    print(f'{name} ready: {url}')
                    return
        except Exception as exc:
            last_error = exc
        time.sleep(2)
    raise TimeoutError(f'{name} failed to start in {timeout_seconds}s: {last_error}')

wait_for_service('Ollama', f'http://{OLLAMA_HOST}:{OLLAMA_PORT}/api/tags', ollama_process, 'ollama.log')
wait_for_service('ComfyUI', f'http://{COMFY_HOST}:{COMFY_PORT}/system_stats', comfy_process, 'comfyui.log')
try:
    import torch
    print(f'CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
except ImportError:
    print('PyTorch is unavailable in this kernel; inspect ComfyUI logs.')
print('Ollama models:', http_json(f'http://{OLLAMA_HOST}:{OLLAMA_PORT}/api/tags'))
print(f'ComfyUI checkpoints: http://{COMFY_HOST}:{COMFY_PORT}/models/checkpoints')

In [ ]:
# Poll manually uploaded JSON requests. Ollama replies use the request filename;
# ComfyUI replies contain that JSON plus a companion image file.
from urllib.parse import urlencode

COMFY_BRIDGE_TIMEOUT = 1800
bridge_stop = threading.Event()
bridge_thread = None
_file_state = {}
_retry_after = {}


def _atomic_json_write(path, data):
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(data, stream, ensure_ascii=False, indent=2)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, path)


def _atomic_bytes_write(path, data):
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as stream:
        stream.write(data)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, path)


def _comfy_get_json(url, timeout=30):
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def _process_comfy_bridge_file(request_file):
    response_file = BRIDGE_OUTPUT / request_file.name
    error_file = BRIDGE_ERRORS / request_file.name
    try:
        with request_file.open("r", encoding="utf-8") as stream:
            payload = json.load(stream)
        if not isinstance(payload, dict) or not isinstance(payload.get("prompt"), dict):
            raise ValueError("Expected a ComfyUI POST /prompt object containing a prompt workflow.")

        for node in payload["prompt"].values():
            if node.get("class_type") == "CheckpointLoaderSimple" and "ckpt_name" in node.get("inputs", {}):
                node["inputs"]["ckpt_name"] = COMFY_CHECKPOINT_FILENAME

        base_url = f"http://{COMFY_HOST}:{COMFY_PORT}"
        queued = http_json(f"{base_url}/prompt", method="POST", payload=payload, timeout=60)
        remote_prompt_id = queued.get("prompt_id")
        if not remote_prompt_id:
            raise RuntimeError(f"ComfyUI did not return prompt_id: {queued!r}")

        deadline = time.monotonic() + COMFY_BRIDGE_TIMEOUT
        image_info = None
        while time.monotonic() < deadline:
            history = _comfy_get_json(f"{base_url}/history/{remote_prompt_id}", timeout=30)
            entry = history.get(remote_prompt_id, {})
            status = entry.get("status", {})
            if status.get("status_str") == "error":
                raise RuntimeError(f"ComfyUI generation failed: {status.get('messages', entry)!r}")
            for node_output in entry.get("outputs", {}).values():
                images = node_output.get("images", [])
                if images:
                    image_info = images[0]
                    break
            if image_info:
                break
            time.sleep(2)
        if not image_info:
            raise TimeoutError(f"ComfyUI job {remote_prompt_id} exceeded {COMFY_BRIDGE_TIMEOUT}s.")

        view_query = urlencode({
            "filename": image_info["filename"],
            "subfolder": image_info.get("subfolder", ""),
            "type": image_info.get("type", "output"),
        })
        with urllib.request.urlopen(f"{base_url}/view?{view_query}", timeout=120) as response:
            image_bytes = response.read()
        if not image_bytes:
            raise RuntimeError(f"ComfyUI returned an empty image for prompt {remote_prompt_id}.")

        extension = Path(image_info["filename"]).suffix.lower() or ".png"
        if extension not in {".png", ".jpg", ".jpeg", ".webp"}:
            raise ValueError(f"Unsupported ComfyUI image extension: {extension}")
        image_filename = request_file.stem + extension
        _atomic_bytes_write(BRIDGE_OUTPUT / image_filename, image_bytes)
        _atomic_json_write(response_file, {
            "status": "completed",
            "image_filename": image_filename,
            "remote_prompt_id": remote_prompt_id,
        })
        error_file.unlink(missing_ok=True)
        print(f"[bridge] ComfyUI completed {request_file.name}; image: {image_filename}")
    except Exception as exc:
        error = {
            "status": "error",
            "error": str(exc),
            "request_file": request_file.name,
            "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        _atomic_json_write(response_file, error)
        _atomic_json_write(error_file, error)
        print(f"[bridge error] ComfyUI {request_file.name}: {exc}; details: {error_file}")


def _handle_bridge_file(request_file):
    if request_file.name.endswith("_comfy_prompt.json"):
        _process_comfy_bridge_file(request_file)
        return

    output_file = BRIDGE_OUTPUT / request_file.name
    error_file = BRIDGE_ERRORS / request_file.name
    if output_file.exists():
        return
    try:
        with request_file.open("r", encoding="utf-8") as stream:
            payload = json.load(stream)
        if not isinstance(payload, dict) or not isinstance(payload.get("messages"), list) or not payload["messages"]:
            raise ValueError("Expected an Ollama /api/chat JSON object with a non-empty messages array.")
        payload["model"] = OLLAMA_MODEL
        payload["stream"] = False
        payload["keep_alive"] = 0
        response = http_json(
            f"http://{OLLAMA_HOST}:{OLLAMA_PORT}/api/chat",
            method="POST",
            payload=payload,
            timeout=600,
        )
        if not isinstance(response, dict) or not isinstance(response.get("message"), dict):
            raise ValueError(f"Unexpected Ollama response: {response!r}")
        _atomic_json_write(output_file, response)
        error_file.unlink(missing_ok=True)
        print(f"[bridge] completed {request_file.name} -> {output_file}")
    except Exception as exc:
        _atomic_json_write(error_file, {
            "error": str(exc),
            "request_file": request_file.name,
            "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        })
        _retry_after[request_file.name] = time.monotonic() + 30
        print(f"[bridge error] {request_file.name}: {exc}; details: {error_file}")


def _bridge_loop(poll_seconds=2, settle_seconds=2):
    while not bridge_stop.is_set():
        try:
            for request_file in sorted(BRIDGE_INPUT.glob("*.json")):
                if (BRIDGE_OUTPUT / request_file.name).exists():
                    continue
                if time.monotonic() < _retry_after.get(request_file.name, 0):
                    continue
                try:
                    stat = request_file.stat()
                except FileNotFoundError:
                    continue
                state = (stat.st_size, stat.st_mtime_ns)
                previous = _file_state.get(request_file.name)
                if previous is None or previous[0] != state:
                    _file_state[request_file.name] = (state, time.monotonic())
                    continue
                if time.monotonic() - previous[1] >= settle_seconds:
                    _handle_bridge_file(request_file)
                    _file_state.pop(request_file.name, None)
        except Exception as exc:
            print(f"[bridge watcher error] {type(exc).__name__}: {exc}")
        bridge_stop.wait(poll_seconds)


def start_bridge_watcher():
    global bridge_thread
    if bridge_thread is not None and bridge_thread.is_alive():
        print("Bridge watcher is already running.")
        return
    bridge_stop.clear()
    bridge_thread = threading.Thread(target=_bridge_loop, name="kaggle-json-bridge", daemon=True)
    bridge_thread.start()
    print(f"Watching {BRIDGE_INPUT}; responses go to {BRIDGE_OUTPUT}.")


def show_service_status():
    for name, process in (("Ollama", ollama_process), ("ComfyUI", comfy_process)):
        state = "running" if process.poll() is None else f"stopped (exit={process.returncode})"
        print(f"{name}: {state}; pid={process.pid}")
    print(f"Bridge watcher: {'running' if bridge_thread and bridge_thread.is_alive() else 'stopped'}")
    print(f"Input: {BRIDGE_INPUT}; responses: {BRIDGE_OUTPUT}; errors: {BRIDGE_ERRORS}")


def stop_services():
    bridge_stop.set()
    if bridge_thread is not None and bridge_thread.is_alive():
        bridge_thread.join(timeout=5)
    for process in (comfy_process, ollama_process):
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=5)
    for log_file in (comfy_log, ollama_log):
        if not log_file.closed:
            log_file.close()
    show_service_status()


start_bridge_watcher()
show_service_status()
print("Upload *_api_chat.json for Ollama, or *_comfy_prompt.json plus workflow payload for ComfyUI.")
print("ComfyUI replies are a same-name JSON and a companion image file in bridge/output.")
print("Copy both to the PC's debug/input; the JSON points to the image filename.")
print(f"ComfyUI API is local to this Kaggle session: http://{COMFY_HOST}:{COMFY_PORT}")
